# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [2]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    if not argv:
        return "Usage: greet <name> | add <x> <y>"

    command = argv[0].lower()

    if command == "greet":
        if len(argv) != 2:
            return "Usage: greet <name>"
        return f"Hello, {argv[1]}!"

    if command == "add":
        if len(argv) != 3:
            return "Usage: add <x> <y>"
        try:
            total = float(argv[1]) + float(argv[2])
            return str(total)
        except ValueError:
            return "Error: add requires numeric values for x and y."

    return "Usage: greet <name> | add <x> <y>"


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input

Hello, Alice!
7.0
Error: add requires numeric values for x and y.


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [3]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]


def build_parser():
    """Return an argparse parser for expense filtering."""
    parser = argparse.ArgumentParser(description="Filter expense records.")
    parser.add_argument("--category", help="Filter by expense category.")
    parser.add_argument("--min", type=float, default=0.0, help="Minimum amount to include.")
    return parser


def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    filtered = []
    for expense in expenses:
        if category is not None and expense.get("category", "").lower() != category.lower():
            continue
        if float(expense.get("amount", 0)) < min_amount:
            continue
        filtered.append(expense)
    return filtered


def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    if not expenses:
        return "No expenses found."

    lines = ["Date       | Category | Amount | Note"]
    for expense in expenses:
        lines.append(
            f"{expense.get('date', ''):<10} | "
            f"{expense.get('category', ''):<8} | "
            f"${expense.get('amount', 0):.2f} | "
            f"{expense.get('note', '')}"
        )
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))

Date       | Category | Amount | Note
2026-10-02 | Food     | $34.20 | Groceries


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [4]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [5]:
from pathlib import Path
from datetime import datetime
import shutil


def make_backup_name(filename):
    """Return a timestamped backup filename."""
    path = Path(filename)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{path.stem}_{timestamp}{path.suffix}"


def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    source = Path(source_path)
    if not source.exists():
        return None

    backup_folder = Path(backup_dir)
    backup_folder.mkdir(exist_ok=True)

    destination = backup_folder / make_backup_name(source.name)
    shutil.copy2(source, destination)
    return str(destination)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)

Backup created: backups/budget_data_20261112_161415.json


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [6]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [7]:
import csv


def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    valid_rows = []
    with open(filename, newline="") as file:
        reader = csv.DictReader(file)
        for row in reader:
            category = (row.get("category") or "").strip()
            amount_raw = (row.get("amount") or "").strip()

            if not category or not amount_raw:
                continue

            try:
                amount = float(amount_raw)
            except ValueError:
                continue

            valid_rows.append({
                "date": (row.get("date") or "").strip(),
                "category": category,
                "amount": amount,
                "note": (row.get("note") or "").strip(),
            })

    return valid_rows


def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0.0) + expense["amount"]
    return totals


def print_summary(summary):
    """Print a formatted category summary."""
    if not summary:
        print("No valid expenses found.")
        return

    longest_name = max(len(str(name)) for name in summary)
    print(f"{'Category':<{longest_name}}  Total")
    for category, total in sorted(summary.items()):
        print(f"{category:<{longest_name}}  ${total:.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)

Category   Total
Food       $39.80
Transport  $2.80


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [8]:
import re


def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    if not 0 <= minute <= 59:
        raise ValueError("Minute must be between 0 and 59.")
    if not 0 <= hour <= 23:
        raise ValueError("Hour must be between 0 and 23.")

    return f"{minute} {hour} * * * python {script_path}"


def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    if not re.fullmatch(r"\d{2}:\d{2}", time_str):
        raise ValueError("Time must be in HH:MM format.")

    hour, minute = map(int, time_str.split(":"))
    if not 0 <= hour <= 23 or not 0 <= minute <= 59:
        raise ValueError("Hour must be between 0 and 23, and minute between 0 and 59.")

    return (
        f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {script_path}" /ST {time_str}'
    )


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))

30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [9]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [10]:
import argparse
import json


def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    try:
        with open(filename, "r") as file:
            data = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

    if not isinstance(data, list):
        return []

    return [item for item in data if isinstance(item, dict)]


def build_suite_parser():
    """Return an argparse parser with subcommands."""
    parser = argparse.ArgumentParser(description="Automation suite for expense data.")
    parser.add_argument("filename", help="Path to the JSON expense file.")

    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list", help="List all expenses.")
    subparsers.add_parser("summary", help="Show totals by category.")
    subparsers.add_parser("backup", help="Create a timestamped backup.")

    return parser


def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    parser = build_suite_parser()
    args = parser.parse_args(argv)

    expenses = load_json_expenses(args.filename)

    if args.command == "list":
        return format_expenses(expenses)

    if args.command == "summary":
        summary = summarize_by_category(expenses)
        if not summary:
            return "No expenses found."

        lines = ["Category | Total"]
        for category, total in sorted(summary.items()):
            lines.append(f"{category} | ${total:.2f}")
        return "\n".join(lines)

    if args.command == "backup":
        result = backup_file(args.filename)
        if result is None:
            return f"File not found: {args.filename}"
        return f"Backup created: {result}"

    return "Invalid command."


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))

Date       | Category | Amount | Note
2026-10-01 | Food     | $18.50 | Lunch
2026-10-02 | Transport | $2.80 | Metro
2026-10-03 | Food     | $21.30 | Dinner
2026-10-04 | Books    | $45.00 | Python book

Category | Total
Books | $45.00
Food | $39.80
Transport | $2.80

Backup created: backups/expenses_suite_20261112_161429.json
